# Work for Final Project Notebook

## Import Data

In [1]:
pip install nbformat


Note: you may need to restart the kernel to use updated packages.


In [29]:
pip install voila

  Using cached jupyter_client-8.8.0-py3-none-any.whl.metadata (8.4 kB)
  Using cached jupyter_server-2.17.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-2.28.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached pyzmq-27.1.0-cp312-abi3-macosx_10_15_universal2.whl.metadata (6.0 kB)
  Using cached tornado-6.5.5-cp39-abi3-macosx_10_9_universal2.whl.metadata (2.8 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jupyter_events-0.12.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached terminado-0.18.1-py3-none-any.whl.metadata (5.8 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached bleach-6.3.0-py3-none-any.whl.metadata (31 kB)
  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
  Using cached jupyterlab_pygments-0.3.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64

In [10]:
#pip install plotly

In [8]:
#pip install pandas

In [14]:
#pip install IPython

In [15]:
#pip install ipywidgets

In [2]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots


## Clean & Set Up Data

In [3]:
# import .tsv as .csv
df = pd.read_csv('2025_specimen_time_series_events_no_phi.tsv', sep='\t')

In [4]:
df.head()

,accession_id,pat_enc_csn_id,pat_mrn_id,barcode,tube_id,specimen_id,test_code,test_performing_dept,test_performing_location,event_street,event_source,event_type,event_dt
0,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,TSH,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
1,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,BASIC,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
2,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,FT4,UCHM,ULAB,Hospital,order,test_ordered_dt,2025-01-07T15:48:00Z
3,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,TSH,UCHM,ULAB,Hospital,order,test_collected_dt,2025-01-07T16:21:00Z
4,8ba983c310,af3cb4efb5,55435a20da,0016c04435,34b5a842cf,5136453571,BASIC,UCHM,ULAB,Hospital,order,test_collected_dt,2025-01-07T16:21:00Z


Reshape to long format (one row per specimen-event) - this way, we can easily compute durations to each event type & compare across specimens

In [5]:
# make sure event_dt is datetime
df['event_dt'] = pd.to_datetime(df['event_dt'])

# pivot to get one column per event type
# if there are multiple events of the same type for a specimen, we take 
# the earliest one (e.g. earliest resulted_dt)
df['test_id'] = df['accession_id'].astype(str) + "_" + df['test_code']

# only use order events to define test_id and timeline, since other events may be missing or duplicated
order_df = df[df['event_source'] == 'order'].copy()

timeline = order_df.pivot_table(
    index='test_id',
    columns='event_type',
    values='event_dt',
    aggfunc='min'  # in case of duplicates, take earliest
).reset_index()

# rename columns for clarity
# prefix with 'test_' to indicate these are test-related events
timeline.columns.name = None
timeline = timeline.rename(columns={
    'collected_dt': 'test_collected_dt',
    'receipt_dt': 'test_receipt_dt',
    'resulted_dt': 'test_min_resulted_dt',
    'verified_dt': 'test_min_verified_dt',
    'cancelled_dt': 'test_cancelled_dt',
    'cancellation_dt': 'test_cancelled_dt'
})

In [6]:
# define start time as earliest available event for each specimen
# use as a reference point to compute durations to other events
event_cols = [
    'test_collected_dt', 'test_receipt_dt',
    'test_min_resulted_dt', 'test_min_verified_dt',
    'test_cancelled_dt'
]
# only consider event columns that actually exist in the data
existing_event_cols = [col for col in event_cols if col in timeline.columns]

# compute start time as min of existing event columns
timeline['start'] = timeline['test_ordered_dt']

# now compute durations in hours
# use only existing event columns to avoid errors
for col in existing_event_cols:
    timeline[col + '_hrs'] = (timeline[col] - timeline['start']).dt.total_seconds() / 3600

In [7]:
timeline.describe()

,test_collected_dt_hrs,test_receipt_dt_hrs,test_min_resulted_dt_hrs,test_min_verified_dt_hrs,test_cancelled_dt_hrs
count,229447.000000,224505.000000,219341.000000,217589.000000,9950.000000
mean,-1.662869,1.427063,3.890947,6.663868,7.796382
std,137.200319,138.256973,38.795072,41.948895,21.059802
min,-45860.666667,-45860.666667,-240.000000,-176.650000,-45.216667
25%,0.000000,0.233333,0.350000,0.550000,0.600000
50%,0.100000,0.716667,1.250000,1.400000,1.750000
75%,0.466667,2.666667,3.983333,4.433333,5.100000
max,354.733333,642.700000,15567.900000,15591.883333,703.433333


In [8]:
# remove rows where collected happens before ordered (bad data? assumption)
timeline = timeline[
    (timeline['test_collected_dt'].isna()) |
    (timeline['test_collected_dt'] >= timeline['test_ordered_dt'])
]

In [9]:
timeline.describe() 

,test_collected_dt_hrs,test_receipt_dt_hrs,test_min_resulted_dt_hrs,test_min_verified_dt_hrs,test_cancelled_dt_hrs
count,182076.000000,177518.000000,173411.000000,173242.000000,8702.000000
mean,0.658631,2.982987,4.913850,5.491968,7.862082
std,2.281814,6.594367,41.266058,42.220810,20.917295
min,0.000000,-0.016667,-16.516667,-0.316667,-2.066667
25%,0.050000,0.516667,0.850000,0.966667,0.716667
50%,0.200000,1.033333,1.616667,1.733333,1.900000
75%,0.616667,3.600000,4.800000,4.933333,5.350000
max,354.733333,642.700000,15567.900000,15591.883333,703.433333


In [10]:
print("Available columns in order_df:", order_df.columns.tolist())

Available columns in order_df: ['accession_id', 'pat_enc_csn_id', 'pat_mrn_id', 'barcode', 'tube_id', 'specimen_id', 'test_code', 'test_performing_dept', 'test_performing_location', 'event_street', 'event_source', 'event_type', 'event_dt', 'test_id']


In [11]:
print(order_df['test_performing_dept'].value_counts().head(20))
print()
print(order_df['test_performing_location'].value_counts().head(20))
print()
print(order_df['event_street'].value_counts().head(10))

test_performing_dept
UCHM         1129305
NHLA          133489
UHEM          129348
UURN          103581
UPOC           60298
UCOA           32425
NIMM           18545
UBB            13907
NCHM            9412
REF1            9210
MICN            2453
VCOR            1361
USP              364
UFLOW            348
UPBG             338
MDCH             248
MIRAV            228
PHEM,UHEM        205
UCHM,UHEM        197
ARUP             190
Name: count, dtype: int64

test_performing_location
ULAB         1410268
NHLA          133489
MM             60522
NCRC           30431
REF1            9210
VCOR            1361
MDCH             248
MIRA             228
NCRC,ULAB        205
ARUP             190
CIN              157
WRD              156
REF3             143
GNDX             139
PROM             127
MAYO             116
ARC              115
REF2              81
BCW               51
MCW               43
Name: count, dtype: int64

event_street
Medical      1019068
Hospital      488987
Plymo

In [12]:
# === DERIVE test_code ===
if 'test_code' not in timeline.columns:
    timeline['test_code'] = timeline['test_id'].str.rsplit('_', n=1).str[-1]

# === WEEKDAY / WEEKEND ===
timeline['order_day'] = timeline['start'].dt.dayofweek
timeline['group'] = timeline['order_day'].apply(
    lambda x: 'Weekend' if x >= 5 else 'Weekday'
)

# === SHIFT ===
timeline['order_hour'] = timeline['start'].dt.hour
timeline['shift'] = timeline['order_hour'].apply(
    lambda h: 'Day (6am–6pm)' if 6 <= h < 18 else 'Night (6pm–6am)'
)

# === DEPARTMENT (location filter) ===
dept_map = order_df[['test_id', 'test_performing_dept']].drop_duplicates('test_id')
timeline = timeline.merge(dept_map, on='test_id', how='left')

# === DEFECT FLAGS ===
timeline['cancelled'] = timeline['test_cancelled_dt'].notna()
timeline['delayed_result'] = timeline['test_min_resulted_dt_hrs'] > 4

# sanity check
print(timeline[['test_id', 'test_code', 'group', 'shift', 'test_performing_dept']].head())

            test_id test_code    group            shift test_performing_dept
0   00007d9883_25HD      25HD  Weekday    Day (6am–6pm)                 UCHM
1  00007d9883_ALSUR     ALSUR  Weekday  Night (6pm–6am)                 UCHM
2   0000b7bd82_25HD      25HD  Weekday    Day (6am–6pm)                 UCHM
3   0000ec2d4a_25HD      25HD  Weekday    Day (6am–6pm)                 UCHM
4   00014dfb63_25HD      25HD  Weekday    Day (6am–6pm)                 UCHM


## Visualizations

### Visualization 1
Visualize (visual, inspectable, graphic timelines) the “average” time series of events for laboratory orders, given a set of dimensional filters (such as “all CBCs ordered in ward CW1”).  Method is open to interpretation, and whichever means the students find most readily available.

In [13]:
# === DASHBOARD CONTROLS ===

test_options = ['All'] + sorted(timeline['test_code'].dropna().unique())
dept_options = ['All'] + sorted(timeline['test_performing_dept'].dropna().unique())
ab_options   = ['Weekday vs Weekend', 'Day Shift vs Night Shift']

test_dd  = widgets.Dropdown(
    options=test_options,
    description='Test Code:',
    value='All'
)
dept_dd  = widgets.Dropdown(
    options=dept_options,
    description='Dept:',
    value='All'
)
ab_radio = widgets.RadioButtons(
    options=ab_options,
    description='Compare By:',
    value='Weekday vs Weekend'
)

In [14]:
# === RENDER FUNCTION ===

# clean label mappings
STAGE_LABELS = {
    'collected_dt': 'Collected',
    'receipt_dt': 'Received at Lab',
    'resulted_dt': 'Result Available',
    'verified_dt': 'Verified',
    'ordered': 'Ordered',
    'collected': 'Collected',
    'received': 'Received at Lab',
    'resulted': 'Result Available',
    'verified': 'Verified',
}

EVENT_LABELS = {
    'cancelled': 'Cancelled',
    'delayed_result': 'Delayed Result (>4hrs)',
}

GROUP_LABELS = {
    'Weekday': 'Weekday',
    'Weekend': 'Weekend',
    'Day (6am–6pm)': 'Day Shift (6am–6pm)',
    'Night (6pm–6am)': 'Night Shift (6pm–6am)',
}

def fmt_hours(h):
    if pd.isna(h):
        return 'N/A'
    hours = int(h)
    minutes = round((h - hours) * 60)
    if hours == 0:
        return f"{minutes}m"
    elif minutes == 0:
        return f"{hours}h"
    else:
        return f"{hours}h {minutes}m"

def render_dashboard(test_code, dept, compare_by):

    # --- apply filters ---
    data = timeline.copy()
    if test_code != 'All':
        data = data[data['test_code'] == test_code]
    if dept != 'All':
        data = data[data['test_performing_dept'] == dept]

    # --- assign A/B group ---
    data = data.copy()
    if compare_by == 'Weekday vs Weekend':
        data['ab_group'] = data['group']
    else:
        data['ab_group'] = data['shift']

    data['ab_group'] = data['ab_group'].map(GROUP_LABELS).fillna(data['ab_group'])

    # --- split into completed vs all ---
    data_completed = data[data['cancelled'] == False].copy()

    n = len(data)
    n_completed = len(data_completed)

    # --- melt for timeline (completed only, no cancelled column) ---
    duration_cols = [c for c in data_completed.columns if c.endswith('_hrs') and 'cancelled' not in c]
    long_df = data_completed.melt(
        id_vars=['test_id', 'ab_group'],
        value_vars=duration_cols,
        var_name='event_stage',
        value_name='hours'
    )
    long_df['event_stage'] = (
        long_df['event_stage']
        .str.replace('_hrs', '', regex=False)
        .str.replace('test_', '', regex=False)
        .str.replace('min_', '', regex=False)
    )
    long_df['event_stage'] = long_df['event_stage'].map(STAGE_LABELS).fillna(long_df['event_stage'])
    long_df['hours'] = long_df['hours'].round(2)

    ab_df = (
        long_df.groupby(['ab_group', 'event_stage'])['hours']
        .mean().round(2).reset_index()
    )
    ab_df['label'] = ab_df['hours'].apply(fmt_hours)

    stage_order = ['Collected', 'Received at Lab', 'Result Available', 'Verified']

    # ================================================================
    # CHART 1: A/B timeline (completed specimens only)
    # ================================================================
    fig_timeline = px.line(
        ab_df, x='event_stage', y='hours',
        color='ab_group', markers=True,
        custom_data=['label'],
        labels={
            'event_stage': 'Specimen Journey Stage',
            'hours': 'Hours from Order',
            'ab_group': 'Group'
        },
        title="Average Specimen Journey Timeline"
    )
    fig_timeline.update_layout(
        hovermode='x unified',
        xaxis={'categoryorder': 'array', 'categoryarray': stage_order}
    )
    fig_timeline.update_traces(
        hovertemplate='%{customdata[0]}'
    )
    fig_timeline.show()

    # ================================================================
    # CHART 2: Event likelihood bar chart (ALL specimens, including cancelled)
    # ================================================================
    prob_df = data.groupby('ab_group').agg(
        cancelled=('cancelled', 'mean'),
        delayed_result=('delayed_result', 'mean')
    ).reset_index().melt(
        id_vars='ab_group',
        var_name='event_type',
        value_name='probability'
    )
    prob_df['probability'] = prob_df['probability'].round(4)
    prob_df['event_type'] = prob_df['event_type'].map(EVENT_LABELS).fillna(prob_df['event_type'])

    fig_prob = px.bar(
        prob_df, x='event_type', y='probability',
        color='ab_group', barmode='group', text_auto='.1%',
        labels={
            'event_type': 'Event Type',
            'probability': 'Likelihood',
            'ab_group': 'Group'
        },
        title="Likelihood of Key Events"
    )
    fig_prob.update_layout(
        yaxis_tickformat=',.0%',
    )
    fig_prob.update_traces(
        hovertemplate='%{y:.1%}'
    )
    fig_prob.show()

    # ================================================================
    # CHART 3: % drop-off between stages (completed specimens only)
    # ================================================================
    stage_cols = [
        ('Ordered',          'test_ordered_dt'),
        ('Collected',        'test_collected_dt'),
        ('Received at Lab',  'test_receipt_dt'),
        ('Result Available', 'test_min_resulted_dt'),
        ('Verified',         'test_min_verified_dt'),
    ]
    stage_cols = [(label, col) for label, col in stage_cols if col in data_completed.columns]

    groups = sorted(data_completed['ab_group'].dropna().unique())

    dropoff_rows = []
    transitions = [
        (f"{stage_cols[i][0]} → {stage_cols[i+1][0]}", stage_cols[i][1], stage_cols[i+1][1])
        for i in range(len(stage_cols) - 1)
    ]

    for grp in groups:
        grp_data = data_completed[data_completed['ab_group'] == grp]
        for label, from_col, to_col in transitions:
            from_count = grp_data[from_col].notna().sum()
            to_count = grp_data[to_col].notna().sum()
            pct_lost = round((1 - to_count / from_count) * 100, 2) if from_count > 0 else 0
            dropoff_rows.append({'Group': grp, 'Transition': label, 'Percent Lost (%)': pct_lost})

    dropoff_df = pd.DataFrame(dropoff_rows)

    fig_dropoff = px.bar(
        dropoff_df,
        x='Transition', y='Percent Lost (%)',
        color='Group', barmode='group',
        text_auto='.2f',
        title="Specimen Drop-off Between Stages"
    )
    fig_dropoff.update_layout(
        xaxis_title='Stage Transition',
        yaxis_title='Specimens Lost (%)',
    )
    fig_dropoff.update_traces(
        hovertemplate='%{y:.2f}%'
    )
    fig_dropoff.show()


widgets.interactive(render_dashboard,
                    test_code=test_dd,
                    dept=dept_dd,
                    compare_by=ab_radio)

interactive(children=(Dropdown(description='Test Code:', options=('All', '$PRVD', '17HP2', '17PRG', '21OH', '2…